# Bacpipe Tutorial
This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---
## 1. Setup & Configuration
Load bacpipe and inspect the current configuration and settings, list all available API endpoints, supported models, and embedding dimensions.

In [ ]:
from IPython.display import display

import os 

# replace this with the path to the directory on your machine
os.chdir('../..')

In [ ]:

import bacpipe

# Load configurations
print('Config: ')
display(bacpipe.config)


# Load more advanced settings
print('Settings: ')
display(bacpipe.settings)


In [ ]:

# Show all bacpipe API endpoints
bacpipe.__all__

In [ ]:


# Get supported models
bacpipe.supported_models

In [ ]:

bacpipe.EMBEDDING_DIMENSIONS

---
## 2. Loading Audio Files
Retrieve all audio files from a directory and extract their datetime information from filenames.

In [ ]:
# Get audio files in directory
audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_as='str'
)
audio_files

In [ ]:
# Extract datetime information from audio filenames
dt_files = []
for file in audio_files:
    dt_files.append(bacpipe.get_dt_filename(file))

dt_files

---
## 3. Ground Truth Labels
Load multi-label ground truth annotations aligned to model timestamps, and generate default labels for a given model and dataset.

In [ ]:
# Get multi-label ground truth fitted to model timestamps
gt = bacpipe.ground_truth_by_model(
    'birdnet',
    audio_dir='bacpipe/tests/test_data',
    annotations_filename='annotations.csv',
    single_label=False
)
gt

In [ ]:
# Generate default labels for a model/dataset combination
dl = bacpipe.create_default_labels(
    'bacpipe/tests/test_data',
    'birdnet'
)
dl

---
## 4. Generating Embeddings — Single File
Create a `Loader` and `Embedder` object manually to generate embeddings for a single audio file. This low-level approach is useful if you don't want bacpipe to save the generated embeddings to disk but rather want to work with them directly in memory.

In [ ]:
# Create a loader object (no folder structure — embeddings are returned directly, not saved)
loader_obj = bacpipe.Loader('bacpipe/tests/test_data')

# Create a model embedder object
embed_obj = bacpipe.Embedder('birdnet', loader_obj)

# Generate embeddings for a single file
embeds = embed_obj.get_embeddings_from_model(loader_obj.files[0])
print('-----')

# Note: loader_obj.embeddings() will be empty as embeddings are not saved in this mode
print("Embeddings can't be retrieved using .embeddings() as they haven't been saved:")
display(loader_obj.embeddings())

print('However, the declared variable still contains the embeddings:')
display(embeds)

print('The embedding object has a classifier object which also contains the predictions:')
display(embed_obj.classifier.predictions)


---
## 5. Generating Embeddings — Full Directory and save results to disk
Process all audio files in a directory using multithreading, saving embeddings and classification predictions to disk.

In [ ]:
# Create loader and embedder with folder structure enabled (embeddings are saved to disk)
loader_obj = bacpipe.Loader(
    'bacpipe/tests/test_data', 
    'birdnet', 
    use_folder_structure=True
    )
embed_obj = bacpipe.Embedder('birdnet', loader_obj)

# Generate embeddings and classification predictions for all audio files
embed_obj.run_inference_pipeline_using_multithreading()

print('-----')
print("Now the embeddings were saved and can be retrieved with .embeddings():")
print(loader_obj.embeddings(return_type='array'))
print("A metadata file of the processing run is saved and also stored in memory:")
print(loader_obj.metadata_dict)
print("The classifier predictions can be retrieved in the same way as the embeddings:")
print(loader_obj.predictions(return_type='array'))

print("A dataframe of all predictions is also saved and accessible through the embedder object:")
print(loader_obj.predictions(return_type='dataframe'))

---
## 6. High-Level Workflow — `generate_embeddings`
Use the `generate_embeddings` convenience function as a simpler alternative to manually constructing `Loader` and `Embedder` objects.

In [ ]:
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data'
)

display(loader_obj.metadata_dict)
display(loader_obj.embeddings())

In [ ]:

audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_as='str'
)

import librosa as lb
import numpy as np

audio = []
for file in audio_files:
    aud, sr = lb.load(file)
    audio.extend(aud)
audio = np.array(audio)
# Create a model embedder object
embed_obj = bacpipe.Embedder('naturebeats')

# Generate embeddings for a single file
embeds = embed_obj.embeddings_using_multithreading(audio)



cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Using device='cuda'
Skipping model.eval() because model is from tensorflow.


---
## 7. Using a Custom Model
Define your own model by subclassing `ModelBaseClass` and plug it directly into bacpipe's pipeline.

In [ ]:
import librosa as lb
from bacpipe.model_pipelines.model_utils import ModelBaseClass

class MyModel(ModelBaseClass):
    SAMPLE_RATE = 48000
    SEGMENT_LENGTH = 48000

    def __init__(self, **kwargs):
        super().__init__(sr=self.SAMPLE_RATE, segment_length=self.SEGMENT_LENGTH, **kwargs)

    def preprocess(self, audio):
        return audio * 10

    def __call__(self, audio):
        audio = audio.cpu().numpy()
        mel_spec = lb.feature.melspectrogram(y=audio, sr=self.SAMPLE_RATE)
        # return array needs to be 2D!
        mel_spec = mel_spec.reshape(
            [len(mel_spec), mel_spec.shape[-2] * mel_spec.shape[-1]]
            )
        return mel_spec

# Low-level approach
loader_obj = bacpipe.Loader('bacpipe/tests/test_data')
embed_obj = bacpipe.Embedder('mymodel', loader_obj, device='cuda', CustomModel=MyModel)
mel_spectrograms = embed_obj.get_embeddings_from_model(loader_obj.files[0])

# High-level approach
loader_obj = bacpipe.generate_embeddings(
    model_name='mymodel',
    audio_dir='bacpipe/tests/test_data',
    CustomModel=MyModel
)

display(loader_obj.metadata_dict)
display(loader_obj.embeddings())


---
## 8. Extending an Existing Model
Subclass an existing bacpipe model to modify its behaviour — for example, squaring the input before passing it through BirdNET.

In [ ]:
from bacpipe.model_pipelines.feature_extractors.birdnet import Model

class MyBirdNETModel(Model):
    def __call__(self, input):
        input = input ** 2
        return self.embeds(input, training=False)

loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet2',
    audio_dir='bacpipe/tests/test_data',
    CustomModel=MyBirdNETModel
)

---
## 9. Full Pipeline — Single Model
Run the complete pipeline for a single model: embedding generation, classification, dimensionality reduction, and visualisation.

In [ ]:
# With a built-in model
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)

# With a custom model
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet2',
    audio_dir='bacpipe/tests/test_data',
    CustomModel=MyBirdNETModel
)

---
## 10. Full Pipeline — Multiple Models
Run the pipeline across multiple models simultaneously, including custom ones, and compare their embeddings side by side.

In [ ]:
# Run pipeline for multiple built-in models
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'naturebeats'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap'
)

display(loader_dictionary['birdnet'].metadata_dict)
display(loader_dictionary['naturebeats'].embeddings())

In [ ]:
# Run pipeline for a mix of built-in and custom models
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'mymodel', 'birdnet2'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap',
    CustomModels=[None, MyModel, MyBirdNETModel],
    check_if_primary_combination_exists=False
)

---
## 11. End-to-End: `bacpipe.play`
The highest-level entry point — runs the full bacpipe pipeline including embeddings, classification, dimensionality reduction, evaluation, and dashboard visualisation in one call.

In [ ]:
bacpipe.play(
    models=['birdnet', 'mymodel', 'birdnet2'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap',
    CustomModels=[None, MyModel, MyBirdNETModel]
)

---
## 12. Benchmarking Classifier Performance
Evaluate a model's classifier against ground truth annotations using precision, recall, and F1 score per species. Labels are matched using exact string lookup with a regex fallback for handling hyphens and spacing differences.

In [ ]:

bacpipe.benchmark(
    'birdnet', 
    'bacpipe/tests/test_data', 
    annotations_file='annotations.csv'
    )

